# 🚀 3D Scene Understanding — Google Colab GPU Server (`scene-compute`)

Notebook này biến Google Colab thành GPU server từ xa cho hệ thống **3D Scene Understanding**.

### Cách hoạt động:
1. Chạy notebook này trên Google Colab với GPU (T4 / L4 / A100).
2. Cloudflare Tunnel tạo một URL HTTPS công khai (ví dụ `https://xxxx.trycloudflare.com`).
3. Điền URL này vào `COLAB_URL` trong file `backend/.env` trên máy local của bạn.
4. Khi bạn upload ảnh và xử lý, máy local sẽ tự động gửi tác vụ GPU (Depth Anything 3, SAM 3, GeoCalib, MetaCLIP 2) sang Colab!

### Bước 1: Kiểm tra GPU
*(Hãy chắc chắn bạn đã vào menu **Runtime** → **Change runtime type** → chọn **T4 GPU** hoặc cao hơn)*

In [ ]:
!nvidia-smi

### Bước 2: Cài đặt `uv` và Python 3.12

In [ ]:
!pip install -q uv
!uv python install 3.12

### Bước 3: Tải mã nguồn dự án
*(Thay đổi URL git bên dưới thành repository của bạn, hoặc upload file zip mã nguồn lên Colab)*

In [ ]:
# Thay bằng repo GitHub của bạn
REPO_URL = "https://github.com/luatducvo/3D-scene.git"

import os
if not os.path.exists("/content/3D-scene"):
    !git clone $REPO_URL /content/3D-scene

%cd /content/3D-scene

### Bước 4: Cài đặt thư viện phụ thuộc cho `scene-compute`
- Chạy với `--extra gpu` để cài đặt đầy đủ Depth Anything 3, SAM 3, GeoCalib, MetaCLIP 2.

In [ ]:
# Cài đặt GPU packages
!uv sync --python 3.12 --package scene-compute --extra gpu

### Bước 5: (Tùy chọn) Tải trước model weights

In [ ]:
import os
os.environ["DATA_DIR"] = "/content/data"
os.environ["MODELS_DIR"] = "/content/models"
os.environ["HF_HOME"] = "/content/models/hf"
os.environ["TORCH_HOME"] = "/content/models/torch"

# Nếu bạn dùng SAM 3 gated weights từ HuggingFace, bỏ comment dòng dưới:
# os.environ["HF_TOKEN"] = "hf_xxxx"

!uv run --python 3.12 python -m scene_compute.download_models

### Bước 6: Khởi chạy Cloudflare Tunnel
Tạo đường hầm HTTPS an toàn để kết nối từ máy tính của bạn về Colab.

In [ ]:
import subprocess, time, re

# Cài đặt cloudflared binary
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Chạy tunnel nền chuyển tiếp port 9000
!pkill cloudflared || true
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:9000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for _ in range(40):
    line = tunnel_proc.stdout.readline()
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break
    time.sleep(0.5)

print("=" * 60)
print("🎉 URL PUBLIC CHO MÁY LOCAL:", public_url)
print("=" * 60)
print("👉 Copy URL trên dán vào `COLAB_URL` trong file `backend/.env` ở máy local của bạn!")

### Bước 7: Khởi chạy `scene-compute` HTTP Server
*(Giữ cell này luôn chạy trong suốt quá trình bạn thao tác trên web app)*

In [ ]:
import os

# Bạn có thể đặt mật khẩu token bất kỳ (nhớ điền khớp vào COMPUTE_TOKEN trong backend/.env)
COMPUTE_TOKEN = "my-secret-token"

os.environ["COMPUTE_TOKEN"] = COMPUTE_TOKEN
os.environ["PORT"] = "9000"
os.environ["DATA_DIR"] = "/content/data"
os.environ["MODELS_DIR"] = "/content/models"
os.environ["HF_HOME"] = "/content/models/hf"
os.environ["TORCH_HOME"] = "/content/models/torch"
os.environ["CHUNK_SIZE_MB"] = "25"

print(f"Starting scene-compute on port 9000 with token: {COMPUTE_TOKEN}")
!uv run --python 3.12 uvicorn scene_compute.server:app --host 0.0.0.0 --port 9000